In [2]:
import numpy as np
import pandas as pd
import geopandas as gpd

In [7]:
import numpy as np
import geopandas as gpd
from pointpats import random as pprandom
from geopandas import points_from_xy
from shapely.geometry import Point, MultiPoint


def sample_with_centroid_primary(row):
    geom = row["Borders"]
    centre = (row["Centroids"].x, row["Centroids"].y)
    size = int(row["F4"]) + int(row["M4"])
    if geom.is_empty or size == 0:
        return MultiPoint()
    pts = pprandom.normal(geom, centre, size=size)
    return points_from_xy(*pts.T).union_all()

def sample_with_centroid_secondary(row):
    geom = row["Borders"]
    centre = (row["Centroids"].x, row["Centroids"].y)
    size = int(row["F11"]) + int(row["M11"])
    if geom.is_empty or size == 0:
        return MultiPoint()
    pts = pprandom.normal(geom, centre, size=size)
    return points_from_xy(*pts.T).union_all()


def distance_based_prefs(
    student_point: Point,
    school_points: gpd.GeoSeries,
    noise_scale: float = 0.0,   # add > 0 to break ties randomly (same units as CRS)
    rng: np.random.Generator | None = None,
) -> np.ndarray:
    """Return student preference lists based on distance to schools with adjustable noise term.

    Args:
        student_point (sp.Point): The student's location.

        school_points (list): School locations (same CRS as student_point).

        noise_scale (float, optional): Standard deviation of Gaussian noise added to distances before ranking.
        Useful to break ties or add mild randomness without destroying proximity
        signal. Set to 0 for deterministic nearest-first ordering. Defaults to 0.0.

    Returns:
        list: List of preferences for each student.
    """
    if student_point:
        distances = np.array([student_point.distance(sp) for sp in list(school_points)])
    else:
        return None
    
    if noise_scale > 0:
        rng = rng or np.random.default_rng()
        distances = distances + rng.normal(0, noise_scale, size=len(distances))

    order = np.argsort(distances)
    return order

def build_student_preferences(
        row,
        school_points: gpd.GeoSeries,
) -> list[list]:
    """Return a list of preference lists, one per student per area.

    Args:
        row (MultiPoint): Student locations.

    Returns:
        list[list]: Preferences of students based on distance to schools.
    """
    points = list(row.geoms)   # individual Points from the MultiPoint
    return np.array([
        distance_based_prefs(pt, school_points, noise_scale=0.0)
        for pt in points
    ])


def distance_based_priorities(
    school_point: Point,
    student_clusters: gpd.GeoSeries,
) -> np.ndarray:
    """_summary_

    Args:
        school_point (Point): _description_
        student_clusters (list[MultiPoint]): _description_
        student_ids (list): _description_
        noise_scale (float, optional): _description_. Defaults to 0.0.
        rng (np.random.Generator | None, optional): _description_. Defaults to None.

    Returns:
        list[list]: _description_
    """
    flat_distances = np.array([
        school_point.distance(student_point)
        for student_points in student_clusters
        for student_point in student_points.geoms
    ])
    # for cluster_geom, cluster_ids in zip(student_clusters, student_ids):
    #     for pt, sid in zip(cluster_geom.geoms, cluster_ids):
    #         flat_ids.append(sid)
    #         flat_distances.append(school_point.distance(pt))

    # flat_distances = np.array(flat_distances)

    order = np.argsort(flat_distances)
    return order

def build_school_priorities(
    school_point: Point,
    student_points: gpd.GeoSeries,
) -> list:
    """
    Add a `priority_list` column to schools_gdf.
    Each entry is a flat list of student IDs ordered nearest-first.
    """
    return np.array([
        distance_based_priorities(school_point, student_points)
    ])

In [4]:
# from src.geo_model.utils import sample_with_centroid_primary, sample_with_centroid_secondary,\
#     build_student_preferences, build_school_priorities

population = pd.read_excel("data/student_data/sapelsoasyoa20222024.xlsx","Mid-2024 LSOA 2021",skiprows=3)
population.rename(columns={"LSOA 2021 Code":"LSOA21CD"},inplace=True)

index_multi_depra = pd.read_csv("data/student_data/File_1_IoD2025 Index of Multiple Deprivation.csv")
index_multi_depra.rename(
    columns={
        "LSOA code (2021)":"LSOA21CD",
        "Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)":"IMD",
        r"Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)":"IMD Decile",
    },
    inplace=True,
)

geoborders = gpd.read_file("data/student_data/LSOA_Boundaries_geospacial_data_2021")
geoborders = geoborders.rename_geometry('Borders')

geocentroids = gpd.read_file("data/student_data/LSOA_PopCentroids_geospatial_data_2021")
geocentroids = geocentroids.rename_geometry('Centroids')

geomerge = geoborders.merge(geocentroids[["LSOA21CD","Centroids"]],"inner","LSOA21CD")
geomerge = geomerge.merge(population[["LSOA21CD","Total","F4","F11","M4","M11"]],
                          "inner","LSOA21CD")
geomerge = geomerge.merge(index_multi_depra[["LSOA21CD","IMD","IMD Decile"]],"inner","LSOA21CD")
geomerge = geomerge[["LSOA21CD","LSOA21NM",
                     "IMD","IMD Decile",
                     "Total","F4","F11","M4","M11",
                     "Centroids","Borders"]]

geo_soton = geomerge[geomerge["LSOA21NM"].str.contains("Southampton")]
geo_soton["Primary Student Locations"] = geo_soton.apply(sample_with_centroid_primary,axis=1)
geo_soton["Secondary Student Locations"] = geo_soton.apply(sample_with_centroid_secondary,axis=1)

In [5]:
school_quality = pd.read_csv(
    "data/school_data/Performancetables_csv/2023-2024/852_ks4final.csv",
    index_col="URN",
    usecols=["URN","P8MEA"],
)

schools = pd.read_csv(
    "data/school_data/edubasealldata20260225.csv",
    index_col="URN",
    usecols=[
        "URN",
        "LSOA (code)","LA (name)",
        "EstablishmentName",
        "TypeOfEstablishment (name)",
        "EstablishmentTypeGroup (name)",
        "PhaseOfEducation (name)",
        "SchoolCapacity",
        "PercentageFSM",
        "Easting",
        "Northing",
        "EstablishmentStatus (name)",
    ],
    encoding="latin-1",
)
schools = schools.merge(school_quality,"inner","URN")
schools = schools.rename(columns={"LSOA (code)":"LSOA21CD"})
schools = schools[schools["EstablishmentStatus (name)"] == "Open"]
schools = schools[schools["EstablishmentTypeGroup (name)"].isin(
    ["Academies",
     "Free Schools",
     "Local authority maintained schools"]
)]
schools = schools[[
    "LSOA21CD","LA (name)",
    "EstablishmentName",
    "TypeOfEstablishment (name)",
    "EstablishmentTypeGroup (name)",
    "PhaseOfEducation (name)",
    "SchoolCapacity",
    "PercentageFSM",
    "P8MEA",
    "Easting",
    "Northing"
]]
schools = gpd.GeoDataFrame(
    schools,
    geometry=gpd.points_from_xy(
        schools.Easting, schools.Northing
    )
)


schools = schools[schools["LA (name)"].isin(["Southampton"])]


primary_schools = schools[schools["PhaseOfEducation (name)"].isin(
    ["All-through",
     "Middle deemed primary",
     "Primary"]
)]
secondary_schools = schools[schools["PhaseOfEducation (name)"].isin(
    ["All-through",
     "Middle deemed secondary",
     "Secondary"]
)]

In [8]:
primary_school_points = primary_schools.geometry

secondary_school_points = secondary_schools.geometry

primary_student_points = geo_soton["Primary Student Locations"].explode()

secondary_student_points = geo_soton["Secondary Student Locations"].explode()

geo_soton["Primary Preferences"] = geo_soton["Primary Student Locations"].apply(
    build_student_preferences,
    school_points=primary_school_points,
)
geo_soton["Secondary Preferences"] = geo_soton["Secondary Student Locations"].apply(
    build_student_preferences,
    school_points=secondary_school_points,
)
schools["Primary Priorities"] = primary_school_points.apply(
    build_school_priorities,
    student_points=geo_soton["Primary Student Locations"],
)
schools["Secondary Priorities"] = secondary_school_points.apply(
    build_school_priorities,
    student_points=geo_soton["Secondary Student Locations"],
)

primary_student_preferences = np.stack(
    geo_soton["Primary Preferences"].explode().dropna().values,
    dtype=np.int32,
)
secondary_student_preferences = np.stack(
    geo_soton["Secondary Preferences"].explode().dropna().values,
    dtype=np.int32,
)

primary_school_priorities = np.vstack(
    schools["Primary Priorities"].dropna().values,
    dtype=np.int32,
)
primary_school_capacities = np.array(
    primary_schools.SchoolCapacity.values,
    dtype=np.int32,
)

secondary_school_priorities = np.vstack(
    schools["Secondary Priorities"].dropna().values,
    dtype=np.int32,
)
secondary_school_capacities = np.array(
    secondary_schools.SchoolCapacity.values,
    dtype=np.int32,
)

In [9]:
def test_function(
        row:pd.Series,
        student_type:str,
        schools:gpd.GeoDataFrame,
):

    imd_decile = row.get("IMD Decile")
    student_points = row.get(student_type)
    student_points = np.array(student_points.geoms)
    school_points = schools.geometry
    school_quality = np.array(schools.P8MEA.values,dtype=np.float32)

    preferences = []
    for student_point in student_points:
        ranking = []
        for i, school_point in enumerate(school_points):
            distance = student_point.distance(school_point)
            quality = school_quality[i]
            quality_factor = np.exp(quality)
            rank = distance/quality_factor
            ranking.append(rank)
            # print(distance,quality,rank,ranking)
        ranking = np.array(ranking)
        preferences.append(ranking.argsort())
        # print(preferences,"plup")
    preferences = np.array(preferences)
    # preferences = np.stack(preferences)
    print(preferences,"blep")

    # print(row)
    # print(distance)
    # print(schools.geometry)
    # print(school_quality)
    print(row.get("Secondary Preferences"))

    return 2*imd_decile

testitem = geo_soton.apply(
    test_function,axis=1,
    student_type="Secondary Student Locations",
    schools=secondary_schools,
)

# testitem

[] blep
[]
[[ 9  8  5  4  2 10  7  6 11  1  3  0]
 [ 9  8  5  4  2 10  7  6 11  1  3  0]
 [ 9  8  5  4  2 10  7  6 11  1  3  0]
 [ 9  8  5  4  2 10  7 11  6  1  3  0]
 [ 9  8  5  4  2 10  7  6 11  1  3  0]
 [ 9  8  5  4  2 10  7 11  6  1  3  0]
 [ 9  5  8  4  2 10  7 11  6  1  3  0]
 [ 9  8  5  4  2 10  7 11  6  1  3  0]
 [ 9  5  8  4  2 10  7 11  6  1  3  0]] blep
[[ 9  0  8  4  2  6 10  1  5 11  7  3]
 [ 9  0  8  4  2  6 10  1  5 11  7  3]
 [ 9  0  8  4  2 10  6  1  5 11  7  3]
 [ 9  0  8  4  2 10  6  1  5 11  7  3]
 [ 9  0  8  4  2 10  6  1  5 11  7  3]
 [ 9  0  8  4  2 10  6  1  5 11  7  3]
 [ 9  0  8  4  2 10  6  1  5 11  7  3]
 [ 9  0  8  4  2 10  6  5  1 11  3  7]
 [ 9  0  8  4  2 10  6  5  1 11  3  7]]
[[ 9  8  5  4  2 10  7  6 11  1  3  0]
 [ 9  8  5  4  2 10  7  6 11  1  3  0]
 [ 9  8  5  4  2 10  7  6 11  1  3  0]
 [ 9  8  5  4  2 10  7  6 11  1  3  0]
 [ 9  8  5  4  2 10  7  6 11  1  3  0]
 [ 9  8  5  4  2 10  7  6 11  1  3  0]
 [ 9  8  5  4  2 10  7  6 11  1  3  0]
 [ 9  8

In [ ]:
# primary_student_preferences = np.load("temp/prefprio.npz")["primary_student_preferences"]

In [ ]:
import pandas as pd

school_quality = pd.read_csv(
    "data/school_data/Performancetables_csv/2023-2024/852_ks4final.csv",
    index_col="URN",
    # usecols=["URN", "SCHNAME", "TOTATT8", "ATT8SCR", "P8MEA"],
)

school_quality[[
    "SCHNAME","P8PUP","P8MEA","P8CILOW","P8CIUPP",
    "P8PUP_FSM6CLA1A","P8MEA_FSM6CLA1A",
    "P8PUP_NFSM6CLA1A","P8MEA_NFSM6CLA1A",
]]

,SCHNAME,P8PUP,P8MEA,P8CILOW,P8CIUPP,P8PUP_FSM6CLA1A,P8MEA_FSM6CLA1A,P8PUP_NFSM6CLA1A,P8MEA_NFSM6CLA1A
URN,,,,,,,,,
116458.0,Bitterne Park School,379,-0.19,-0.34,-0.05,113,-0.37,266,-0.12
116469.0,Cantell School,224,0.21,0.02,0.39,80,-0.36,144,0.52
116622.0,The Cedar School,NE,NE,NE,NE,SUPP,SUPP,SUPP,SUPP
144205.0,Great Oaks School,49,-1.58,-1.98,-1.19,24,-1.58,25,-1.58
116568.0,The Gregg School,NP,NP,NP,NP,NP,NP,NP,NP
116580.0,King Edward VI School,NP,NP,NP,NP,NP,NP,NP,NP
135628.0,Oasis Academy Lord's Hill,149,-0.91,-1.14,-0.68,72,-1.26,77,-0.59
135629.0,Oasis Academy Mayfield,175,-0.37,-0.58,-0.16,63,-0.88,112,-0.08
146284.0,Oasis Academy Sholing,194,-0.66,-0.86,-0.46,72,-1.13,122,-0.39


In [ ]:
import pandas as pd

routes = pd.read_csv(
    "data/route_data/198Stops.csv",
    index_col="ATCOCode",
    usecols=["ATCOCode","CommonName","Easting","Northing"]
)

routes.head()

,CommonName,Easting,Northing
ATCOCode,,,
1980SNA90772,Henstead Road,441851,112648
1980SNA90870,Henstead Road,441812,112695
1980BITERNE0,Bitterne Rail Station,443891,113406
1980MBRK0,Millbrook (Hants) Rail Station,439882,112625
1980REDBDGE0,Redbridge (Hants) Rail Station,437340,113520


In [ ]:
import numpy as np
from numba import njit

@njit
def fast_DAT(
        student_preferences:np.ndarray[(3,), np.int32],
        school_priorities:np.ndarray[(3,), np.int32],
        school_capacities:np.ndarray[(1,), np.int32],
        route_capacities:np.ndarray[(1,), np.int32],
) -> np.ndarray:
    """Fast Deferred Acceptance with Transportation.
    Takes student preferences, school priorities, and
    school and route capacities, then outputs a matching.

    Args:
        student_preferences (np.ndarray[student, preference, object]): Elements of the array are schools and routes.
        school_priorities (np.ndarray[school, route, student]): Elements of the array are school priority rankings.
        school_capacities (np.ndarray[school]): Elements of the array are school capacities.
        route_capacities (np.ndarray[route]): Elements of the array are route capacities.

    Returns:
        np.ndarray[student, object]: Final matching of students to schools and/or routes.
    """
    n_students, max_preferences, _ = student_preferences.shape
    n_schools = school_priorities.shape[0]
    n_routes = route_capacities.shape[0]
    matching = np.full((n_students, 2), -1, dtype=np.int32)

    student_next_preference_idx = np.zeros(n_students, dtype=np.int32)
    school_acceptance_numbers = np.zeros(n_schools, dtype=np.int32)
    route_acceptance_numbers = np.zeros(n_routes, dtype=np.int32)
    assigned_students = np.full((n_schools, n_routes+1, n_students), -1, dtype=np.int32)

    free_students = np.arange(n_students, dtype=np.int32)
    pointer = n_students
    while pointer > 0:
        # print(pointer,"\n")
        pointer -= 1
        s_id = free_students[pointer]
        s_rank = student_next_preference_idx[s_id]
        if s_rank >= max_preferences:
            continue
        student_next_preference_idx[s_id] += 1

        target_school = student_preferences[s_id, s_rank, 0]
        target_route = student_preferences[s_id, s_rank, 1]
        if target_school < 0 or target_school >= n_schools:
            free_students[pointer] = s_id
            pointer += 1
            continue

        c_rank = school_priorities[target_school, target_route, s_id]
        c_acc_num = school_acceptance_numbers[target_school]
        school_is_full = c_acc_num >= school_capacities[target_school]
        if target_route > -1:
            r_acc_num = route_acceptance_numbers[target_route]
            route_is_full = r_acc_num >= route_capacities[target_route]
        else: route_is_full = False
        accepted = False

        if not school_is_full and not route_is_full:
            assigned_students[target_school, target_route, s_id] = c_rank
            school_acceptance_numbers[target_school] += 1
            if target_route > -1:
                route_acceptance_numbers[target_route] += 1
            matching[s_id, 0] = target_school
            matching[s_id, 1] = target_route
            accepted = True

        elif route_is_full:
            lowest_priority_student = assigned_students[target_school, target_route].argmax()
            eviction_required = (
                c_rank < school_priorities[target_school, target_route, lowest_priority_student]
            )
            accepted = False
            if eviction_required:
                assigned_students[target_school, target_route, lowest_priority_student] = -1
                assigned_students[target_school, target_route, s_id] = c_rank
                matching[lowest_priority_student, 0] = -1
                matching[lowest_priority_student, 1] = -1
                matching[s_id, 0] = target_school
                matching[s_id, 1] = target_route
                free_students[pointer] = lowest_priority_student
                pointer += 1
                accepted = True

        else:
            lowest_priority_student = -1
            lowest_priority_route = -1
            worst_rank = -1
            for r in range(n_routes + 1):
                for s in range(n_students):
                    rank = assigned_students[target_school, r, s]
                    if rank > worst_rank:
                        worst_rank = rank
                        lowest_priority_route = r
                        lowest_priority_student = s
            # lowest_priority_column = assigned_students[target_school].max(axis=1)
            # lowest_priority_route = lowest_priority_column.argmax()
            # lowest_priority_student = assigned_students[target_school, lowest_priority_route].argmax()
            eviction_required = (
                c_rank < worst_rank
            )
            accepted = False
            if eviction_required:
                assigned_students[target_school, lowest_priority_route, lowest_priority_student] = -1
                assigned_students[target_school, target_route, s_id] = c_rank
                matching[lowest_priority_student, 0] = -1
                matching[lowest_priority_student, 1] = -1
                matching[s_id, 0] = target_school
                matching[s_id, 1] = target_route
                free_students[pointer] = lowest_priority_student
                pointer += 1
                accepted = True

        if not accepted:
            free_students[pointer] = s_id
            pointer += 1
    return matching



fast_DAT(
    np.array([
        [
            [0,0],
            [1,-1],
            [1,1],
        ],
        [
            [0,0],
            [0,-1],
            [1,-1],
        ],
        [
            [1,1],
            [0,-1],
            [1,-1],
        ],
        [
            [1,-1],
            [0,-1],
            [-1,-1],
        ],
    ]),
    np.array([
        [
            [ 0, 1, 2, 3],
            [-1,-1,-1,-1],
            [ 4, 5, 6, 7],
        ],
        [
            [-1,-1,-1,-1],
            [ 3, 2, 1, 0],
            [ 4, 5, 6, 7],
        ],
    ]),
    np.array([1,1]),
    np.array([1,1])
)

array([[ 0,  0],
       [-1, -1],
       [ 1,  1],
       [-1, -1]], dtype=int32)